# Savora — Conversational AI + Backend Integration Contracts (Team Review Edition)

**Based on the supplied Savora repository snapshot and the existing notebook.** This notebook is an executable reference and evaluation harness, **not** a substitute for backend migrations, deployed Next.js screens, production KB approvals, trained ranking models or a weekly CSP solver. All new APIs below are **proposed** until jointly approved.

## 1. Repository audit (observed, not assumed)

| Requirement | Uploaded repository status | This notebook |
|---|---|---|
| Profile schema | Backend `nutrition_profiles` package is empty; no final API DTO | Proposed validated contract |
| Profile to reasoning context | `RuleContext` exists with strict digest/version rules; mapping from UI profile absent | Boundary + explicit adapter obligation |
| Recipe filtering | `filter_candidates` implemented | Typed adapter |
| Combination evaluation | `evaluate_combination` implemented | Typed adapter |
| Daily CSP | `solve_daily_plan` implemented (3 slots, bounded search) | Typed adapter |
| Weekly 21-meal CSP | No weekly solver found in uploaded snapshot | No fake solver; demonstration only |
| Accepted KB, source identity | Knowledge snapshots, manifests, digest checking implemented in AI layer | Trusted release registry boundary |
| Backend endpoints | FastAPI skeleton, health only; no meal-plan endpoints | Proposed request/response contracts |
| Storage, swaps, regeneration | No repository service/DB implementation found | Local SQLite reference implementation |

Existing AI code: `apps/ai_system/src/savora_ai/core/reasoning/` and `apps/ai_system/src/savora_ai/schemas/`. Backend skeleton: `apps/backend/src/savora_backend/`. Proposed backend locations: `core/nutrition_profiles/`, `core/meal_plans/`, `contexts/` and router modules to be created. These are **proposals**, not existing routes.

## 2. Validated profile and planning inputs

In [1]:
from __future__ import annotations

import json
import sqlite3
from dataclasses import asdict, dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from uuid import uuid4

MEAL_SLOTS = ('breakfast', 'lunch', 'dinner')
VALID_DIETS = {'omnivore', 'vegetarian', 'vegan', 'pescatarian'}
VALID_ALLERGENS = {'milk', 'eggs', 'fish', 'shellfish', 'tree_nuts', 'peanuts', 'wheat', 'soy', 'sesame'}  # noqa: E501

def require_string_list(value: Any, label: str) -> list[str]:
    if not isinstance(value, list) or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError(f'{label} must be a list of nonempty strings')
    return list(dict.fromkeys(x.strip().lower() for x in value))

@dataclass
class UserProfile:
    user_id: str
    dietary_preference: str = 'omnivore'
    restrictions: list[str] = field(default_factory=list)
    allergies: list[str] = field(default_factory=list)
    disliked_ingredients: list[str] = field(default_factory=list)
    preferred_cuisines: list[str] = field(default_factory=list)
    calorie_target_kcal: int | None = None
    protein_target_g: float | None = None
    carb_target_g: float | None = None
    fat_target_g: float | None = None
    max_prep_minutes: int | None = None
    budget_per_day: float | None = None
    household_size: int = 1
    timezone: str = 'UTC'

    def __post_init__(self):
        if not isinstance(self.user_id, str) or not self.user_id.strip():
            raise ValueError('user_id is required')
        if self.dietary_preference not in VALID_DIETS:
            raise ValueError(f'Unsupported dietary_preference: {self.dietary_preference}')
        for name in ('restrictions', 'allergies', 'disliked_ingredients', 'preferred_cuisines'):
            setattr(self, name, require_string_list(getattr(self, name), name))
        if not set(self.allergies).issubset(VALID_ALLERGENS):
            raise ValueError('Unknown allergen; use canonical allergen names')
        for name in ('calorie_target_kcal', 'protein_target_g', 'carb_target_g', 'fat_target_g', 'max_prep_minutes', 'budget_per_day'):  # noqa: E501
            v = getattr(self, name)
            if v is not None and (isinstance(v, bool) or not isinstance(v, (int, float)) or (not 0 < v < float('inf'))):  # noqa: E501
                raise ValueError(f'{name} must be a finite positive number')
        if not isinstance(self.household_size, int) or isinstance(self.household_size, bool) or self.household_size < 1:  # noqa: E501
            raise ValueError('household_size must be a positive integer')
        if not self.timezone:
            raise ValueError('timezone is required')
profile = UserProfile(user_id='demo-user', dietary_preference='vegetarian', allergies=['peanuts'], calorie_target_kcal=2100, protein_target_g=90, max_prep_minutes=45)  # noqa: E501
print(json.dumps(asdict(profile), indent=2))


{
  "user_id": "demo-user",
  "dietary_preference": "vegetarian",
  "restrictions": [],
  "allergies": [
    "peanuts"
  ],
  "disliked_ingredients": [],
  "preferred_cuisines": [],
  "calorie_target_kcal": 2100,
  "protein_target_g": 90,
  "carb_target_g": null,
  "fat_target_g": null,
  "max_prep_minutes": 45,
  "budget_per_day": null,
  "household_size": 1,
  "timezone": "UTC"
}


## 3. Accepted knowledge releases and trusted source registry

The real AI layer uses cryptographically bound snapshot models (`KnowledgeBaseSnapshot`, `RuleContext`, candidate and daily-source snapshots). **Never create a fake digest to satisfy their validators.** Production loaders must select an approved KB by ID/version/digest from a trusted database or release registry, then deserialize and validate it using the real Pydantic DTOs. Do not trust a client-supplied KB body or digest.

In [2]:
@dataclass(frozen=True)
class ReleaseIdentity:
    kb_id: str
    kb_version: str
    kb_digest: str
    source_id: str
    source_version: str

    def __post_init__(self):
        for key, val in asdict(self).items():
            if not isinstance(val, str) or not val.strip():
                raise ValueError(f'{key} must be nonempty')

class ApprovedReleaseRegistry:

    def __init__(self):
        self._approved: dict[tuple[str, str], ReleaseIdentity] = {}

    def approve(self, identity: ReleaseIdentity) -> None:
        self._approved[identity.kb_id, identity.kb_version] = identity

    def resolve(self, kb_id: str, version: str, expected_digest: str) -> ReleaseIdentity:
        record = self._approved.get((kb_id, version))
        if record is None or record.kb_digest != expected_digest:
            raise ValueError('Unapproved knowledge release or digest mismatch')
        return record
registry = ApprovedReleaseRegistry()
demo_release = ReleaseIdentity('demo-kb', 'demo-v1', 'demo-digest-not-a-real-kb-hash', 'demo-source', 'v1')  # noqa: E501
registry.approve(demo_release)
assert registry.resolve(demo_release.kb_id, demo_release.kb_version, demo_release.kb_digest) == demo_release  # noqa: E501
print('Local release registry validation passed (demo only; not an accepted production KB).')


Local release registry validation passed (demo only; not an accepted production KB).


## 4. API request/response contracts (proposed)

Suggested routes: `PUT /v1/profiles/{user_id}`, `POST /v1/meal-plans`, `GET /v1/meal-plans/{plan_id}`, `POST /v1/meal-plans/{plan_id}/swaps`, `POST /v1/meal-plans/{plan_id}/regenerate`. They are **not currently implemented** in the uploaded FastAPI skeleton. Backend must derive authenticated user identity server-side and enforce plan ownership.

In [3]:
@dataclass
class PlanRequest:
    user_id: str
    start_date: str
    days: int
    profile: UserProfile
    release: ReleaseIdentity
    mode: str = 'weekly'
    exclude_recipe_ids: list[str] = field(default_factory=list)

    def __post_init__(self):
        from datetime import date
        date.fromisoformat(self.start_date)
        if self.days != 7 or self.mode != 'weekly':
            raise ValueError('21-meal contract requires exactly seven days')
        if self.user_id != self.profile.user_id:
            raise ValueError('Profile owner and plan owner mismatch')
        self.exclude_recipe_ids = require_string_list(self.exclude_recipe_ids, 'exclude_recipe_ids')

@dataclass
class MealAssignment:
    day: int
    slot: str
    recipe_id: str
    recipe_name: str
    status: str
    explanation: str

    def __post_init__(self):
        if self.day not in range(1, 8) or self.slot not in MEAL_SLOTS:
            raise ValueError('Invalid day or slot')
        if not self.recipe_id or not self.recipe_name or (not self.explanation):
            raise ValueError('Missing meal assignment fields')
        if self.status not in {'pass', 'fail', 'unknown', 'not_evaluated'}:
            raise ValueError('Invalid constraint status')

@dataclass
class PlanResponse:
    plan_id: str
    user_id: str
    revision: int
    status: str
    release: ReleaseIdentity
    meals: list[MealAssignment]
    constraint_results: list[dict[str, Any]]
    explanations: list[str]
    generated_at: str
    validated_by_csp: bool = False

    def __post_init__(self):
        if len(self.meals) != 21 or len({(m.day, m.slot) for m in self.meals}) != 21:
            raise ValueError('Weekly plan must contain 21 unique day/slot assignments')
        if self.validated_by_csp and any(m.status != 'pass' for m in self.meals):
            raise ValueError('CSP-validated plan cannot contain non-passing meals')
request = PlanRequest(user_id=profile.user_id, start_date='2026-10-19', days=7, profile=profile, release=demo_release)  # noqa: E501
print(json.dumps({'endpoint': 'POST /v1/meal-plans', 'request_example': asdict(request)}, indent=2))


{
  "endpoint": "POST /v1/meal-plans",
  "request_example": {
    "user_id": "demo-user",
    "start_date": "2026-10-19",
    "days": 7,
    "profile": {
      "user_id": "demo-user",
      "dietary_preference": "vegetarian",
      "restrictions": [],
      "allergies": [
        "peanuts"
      ],
      "disliked_ingredients": [],
      "preferred_cuisines": [],
      "calorie_target_kcal": 2100,
      "protein_target_g": 90,
      "carb_target_g": null,
      "fat_target_g": null,
      "max_prep_minutes": 45,
      "budget_per_day": null,
      "household_size": 1,
      "timezone": "UTC"
    },
    "release": {
      "kb_id": "demo-kb",
      "kb_version": "demo-v1",
      "kb_digest": "demo-digest-not-a-real-kb-hash",
      "source_id": "demo-source",
      "source_version": "v1"
    },
    "mode": "weekly",
    "exclude_recipe_ids": []
  }
}


## 5. AI engine adapters — actual repository signatures

These adapters import the repository's real Pydantic contracts and functions. They **require genuine validated** knowledge/context/candidate/source objects, and intentionally do not synthesize them from incomplete profile information. Profile-to-`RuleContext` mapping must be implemented by the knowledge/context owner after canonical facts and accepted release identities are agreed.

In [4]:
def find_repo_root(start: Path | None=None) -> Path | None:
    for candidate in [start or Path.cwd(), *(start or Path.cwd()).parents]:
        if (candidate / 'apps' / 'ai_system' / 'src' / 'savora_ai').is_dir():
            return candidate
    return None

def load_real_ai_interfaces(repo_root: Path | None=None) -> dict[str, Any]:
    import sys
    root = repo_root or find_repo_root()
    if root is None:
        raise FileNotFoundError('Open this notebook from inside the cloned Savora repository')
    ai_src = root / 'apps' / 'ai_system' / 'src'
    if str(ai_src) not in sys.path:
        sys.path.insert(0, str(ai_src))
    from savora_ai.core.reasoning.combinations import evaluate_combination
    from savora_ai.core.reasoning.daily import build_day_check_manifest, solve_daily_plan
    from savora_ai.core.reasoning.filtering import filter_candidates
    from savora_ai.core.reasoning.kb import build_knowledge_snapshot
    from savora_ai.schemas.candidates import CandidateRecipeFacts, RuleContext
    from savora_ai.schemas.combinations import CombinationFacts
    from savora_ai.schemas.daily import DailyPlanningRequest, DailySourceSnapshot
    from savora_ai.schemas.daily_solver import DailySearchLimits
    from savora_ai.schemas.knowledge_base import KnowledgeBaseSnapshot
    return dict(filter_candidates=filter_candidates, evaluate_combination=evaluate_combination, solve_daily_plan=solve_daily_plan, build_day_check_manifest=build_day_check_manifest, build_knowledge_snapshot=build_knowledge_snapshot, RuleContext=RuleContext, CandidateRecipeFacts=CandidateRecipeFacts, CombinationFacts=CombinationFacts, DailyPlanningRequest=DailyPlanningRequest, DailySourceSnapshot=DailySourceSnapshot, DailySearchLimits=DailySearchLimits, KnowledgeBaseSnapshot=KnowledgeBaseSnapshot)  # noqa: E501

def run_recipe_filter(candidates, knowledge, context, interfaces):
    return interfaces['filter_candidates'](tuple(candidates), knowledge=knowledge, context=context)

def run_combination_evaluation(combination, candidates, knowledge, context, interfaces):
    return interfaces['evaluate_combination'](combination, candidates=tuple(candidates), knowledge=knowledge, context=context)  # noqa: E501

def run_daily_csp(request, candidates, knowledge, context, sources, interfaces, limits=None):
    kwargs = dict(candidates=tuple(candidates), knowledge=knowledge, context=context, sources=sources)  # noqa: E501
    if limits is not None:
        kwargs['limits'] = limits
    return interfaces['solve_daily_plan'](request, **kwargs)
root = find_repo_root()
if root is None:
    print('Standalone contract demo mode. Open notebook inside repository to enable real AI imports.')  # noqa: E501
else:
    try:
        ai_interfaces = load_real_ai_interfaces(root)
        print('Real Savora AI interfaces imported:', sorted(ai_interfaces))
    except (ImportError, ModuleNotFoundError) as exc:
        print('Real AI import unavailable; install project dependencies:', exc)


Real Savora AI interfaces imported: ['CandidateRecipeFacts', 'CombinationFacts', 'DailyPlanningRequest', 'DailySearchLimits', 'DailySourceSnapshot', 'KnowledgeBaseSnapshot', 'RuleContext', 'build_day_check_manifest', 'build_knowledge_snapshot', 'evaluate_combination', 'filter_candidates', 'solve_daily_plan']


## 6. How profile data enters AI reasoning (required integration boundary)

**Flow:** authenticated user → validate `UserProfile` → fetch approved KB release and source snapshots → map dietary/allergy/targets to canonical, evidence-backed facts → construct validated `RuleContext` with KB digest and context version → recipe filtering → combination evaluation → daily CSP → **weekly coordinator (not yet implemented)** → explanations/results → persistent storage.

Safety rule: unknown allergens, absent nutrition facts, or unresolved hard constraints must **not** be treated as passing. A weekly result is publishable only after all 21 assignments have been evaluated and required day/week constraints have passed. The repository currently has day-level reasoning, not an authoritative seven-day optimizer.

In [5]:
def require_authoritative_weekly_solver(solver=None):
    if solver is None:
        raise NotImplementedError('No authoritative weekly 21-meal CSP solver was found in the uploaded repository. Do not publish the demonstration plan as constraint-validated.')  # noqa: E501
    return solver
print('Weekly CSP status: NOT IMPLEMENTED in supplied repository snapshot')


Weekly CSP status: NOT IMPLEMENTED in supplied repository snapshot


## 7. Contract-only 21-meal demonstration

The following uses placeholder recipe IDs and **not_evaluated** statuses solely to test serialization, uniqueness, persistence and swaps. It is not a recommendation or safe allergy/nutrition plan.

In [6]:
demo_meals = [MealAssignment(day=d, slot=s, recipe_id=f'placeholder-{d}-{s}', recipe_name=f'Example {s.title()} {d}', status='not_evaluated', explanation='Contract placeholder: no recipe facts or constraints evaluated') for d in range(1, 8) for s in MEAL_SLOTS]  # noqa: E501
demo_plan = PlanResponse(plan_id=str(uuid4()), user_id=profile.user_id, revision=1, status='draft_unvalidated', release=demo_release, meals=demo_meals, constraint_results=[{'status': 'not_evaluated', 'scope': 'weekly'}], explanations=['No weekly CSP run; no allergy or nutrition validation performed'], generated_at=datetime.now(UTC).isoformat(), validated_by_csp=False)  # noqa: E501
assert len(demo_plan.meals) == 21
print('Generated 21 contract placeholders; NOT a validated meal plan.')


Generated 21 contract placeholders; NOT a validated meal plan.


## 8. Storage, swap and regeneration (local reference only)

SQLite below is an isolated demo persistence layer. Production should use backend DB migrations, transactions, authenticated ownership checks, concurrency control, and full audit trails. Swap proposals must be re-evaluated by the actual reasoning engine before being accepted; the demo keeps them unvalidated.

In [7]:
class LocalPlanStore:

    def __init__(self, database_path=':memory:'):
        self.connection = sqlite3.connect(database_path)
        self.connection.execute('CREATE TABLE IF NOT EXISTS plans (plan_id TEXT PRIMARY KEY, user_id TEXT NOT NULL, revision INTEGER NOT NULL, payload TEXT NOT NULL)')  # noqa: E501
        self.connection.commit()

    def save(self, plan: PlanResponse):
        payload = json.dumps(asdict(plan), allow_nan=False)
        with self.connection:
            existing = self.connection.execute('SELECT revision FROM plans WHERE plan_id=?', (plan.plan_id,)).fetchone()  # noqa: E501
            if existing is not None and plan.revision <= existing[0]:
                raise ValueError('Revision must increase to prevent stale writes')
            self.connection.execute('INSERT INTO plans(plan_id,user_id,revision,payload) VALUES(?,?,?,?) ON CONFLICT(plan_id) DO UPDATE SET revision=excluded.revision,payload=excluded.payload', (plan.plan_id, plan.user_id, plan.revision, payload))  # noqa: E501

    def get(self, plan_id: str, user_id: str) -> PlanResponse:
        row = self.connection.execute('SELECT payload FROM plans WHERE plan_id=? AND user_id=?', (plan_id, user_id)).fetchone()  # noqa: E501
        if row is None:
            raise KeyError('Plan not found for this user')
        data = json.loads(row[0])
        data['release'] = ReleaseIdentity(**data['release'])
        data['meals'] = [MealAssignment(**m) for m in data['meals']]
        return PlanResponse(**data)

def propose_swap(plan: PlanResponse, day: int, slot: str, replacement_id: str, replacement_name: str) -> PlanResponse:  # noqa: E501
    if not replacement_id.strip() or not replacement_name.strip():
        raise ValueError('Replacement ID/name required')
    data = asdict(plan)
    matches = [m for m in data['meals'] if m['day'] == day and m['slot'] == slot]
    if len(matches) != 1:
        raise ValueError('Unknown day/slot')
    target = matches[0]
    target.update(recipe_id=replacement_id, recipe_name=replacement_name, status='not_evaluated', explanation='Swap proposed; requires full recipe/day/week constraint re-evaluation')  # noqa: E501
    data['revision'] += 1
    data['status'] = 'draft_unvalidated'
    data['validated_by_csp'] = False
    data['constraint_results'] = [{'scope': 'weekly', 'status': 'not_evaluated'}]
    data['explanations'] = ['Swap invalidates previous validation; rerun reasoning before publishing']  # noqa: E501
    data['generated_at'] = datetime.now(UTC).isoformat()
    data['release'] = ReleaseIdentity(**data['release'])
    data['meals'] = [MealAssignment(**m) for m in data['meals']]
    return PlanResponse(**data)

def request_regeneration(old_plan: PlanResponse, exclude_recipe_ids: list[str]) -> dict:
    return {'user_id': old_plan.user_id, 'previous_plan_id': old_plan.plan_id, 'expected_revision': old_plan.revision, 'exclude_recipe_ids': require_string_list(exclude_recipe_ids, 'exclude_recipe_ids'), 'status': 'pending_solver', 'message': 'Invoke authoritative planner; do not reuse stale validations'}  # noqa: E501
store = LocalPlanStore()
store.save(demo_plan)
assert store.get(demo_plan.plan_id, profile.user_id).revision == 1
swapped = propose_swap(demo_plan, 1, 'lunch', 'new-placeholder', 'Example replacement')
store.save(swapped)
assert store.get(demo_plan.plan_id, profile.user_id).revision == 2
print('Stored and retrieved draft; swap increments revision and invalidates constraints')
print(json.dumps(request_regeneration(swapped, ['placeholder-1-lunch']), indent=2))


Stored and retrieved draft; swap increments revision and invalidates constraints
{
  "user_id": "demo-user",
  "previous_plan_id": "ae4fb656-2406-4466-9dbc-c48a550b8df4",
  "expected_revision": 2,
  "exclude_recipe_ids": [
    "placeholder-1-lunch"
  ],
  "status": "pending_solver",
  "message": "Invoke authoritative planner; do not reuse stale validations"
}


## 9. Example responses and proposed backend wiring

`POST /v1/meal-plans` should return `202 Accepted` with a job ID if asynchronous, or `201 Created` only after an authoritative plan exists. `GET` returns plan status and the 21 assignments only for complete plans; incomplete solver runs should return diagnostics, not invented meals. `POST /swaps` takes `{day, slot, replacement_recipe_id, expected_revision}` and rechecks all affected hard constraints. `POST /regenerate` takes `{expected_revision, excluded_recipe_ids, changed_profile?}` and creates a new revision/job. Return `409` on revision conflicts, `422` on invalid inputs or infeasible constraints, `403` on ownership violations. **These are proposed contracts, not live endpoints.**

In [8]:
proposed_contracts = {'PUT /v1/profiles/{user_id}': {'request': 'UserProfile', 'response': 'UserProfile + version'}, 'POST /v1/meal-plans': {'request': 'PlanRequest', 'response': '202 {job_id, status} or 201 PlanResponse after verified CSP'}, 'GET /v1/meal-plans/{plan_id}': {'response': 'PlanResponse (if complete) or job status/diagnostics'}, 'POST /v1/meal-plans/{plan_id}/swaps': {'request': {'day': 1, 'slot': 'lunch', 'replacement_recipe_id': 'recipe-id', 'expected_revision': 2}, 'response': 'revalidated PlanResponse or 422/409'}, 'POST /v1/meal-plans/{plan_id}/regenerate': {'request': {'expected_revision': 2, 'excluded_recipe_ids': []}, 'response': '202 {job_id, status}'}}  # noqa: E501
print(json.dumps(proposed_contracts, indent=2))


{
  "PUT /v1/profiles/{user_id}": {
    "request": "UserProfile",
    "response": "UserProfile + version"
  },
  "POST /v1/meal-plans": {
    "request": "PlanRequest",
    "response": "202 {job_id, status} or 201 PlanResponse after verified CSP"
  },
  "GET /v1/meal-plans/{plan_id}": {
    "response": "PlanResponse (if complete) or job status/diagnostics"
  },
  "POST /v1/meal-plans/{plan_id}/swaps": {
    "request": {
      "day": 1,
      "slot": "lunch",
      "replacement_recipe_id": "recipe-id",
      "expected_revision": 2
    },
    "response": "revalidated PlanResponse or 422/409"
  },
  "POST /v1/meal-plans/{plan_id}/regenerate": {
    "request": {
      "expected_revision": 2,
      "excluded_recipe_ids": []
    },
    "response": "202 {job_id, status}"
  }
}


## 10. Automated contract smoke tests

In [9]:
def expect_error(fn, error_type=ValueError):
    try:
        fn()
    except error_type:
        return
    raise AssertionError(f'Expected {error_type.__name__}')
expect_error(lambda: UserProfile(user_id='x', allergies=['unknown_allergen']))
expect_error(lambda: UserProfile(user_id='x', dietary_preference='invalid'))
expect_error(lambda: UserProfile(user_id='x', calorie_target_kcal=-1))
expect_error(lambda: PlanRequest(user_id='wrong', start_date='2026-10-19', days=7, profile=profile, release=demo_release))  # noqa: E501
expect_error(lambda: registry.resolve('demo-kb', 'demo-v1', 'wrong'))
expect_error(lambda: PlanResponse(plan_id='x', user_id='x', revision=1, status='complete', release=demo_release, meals=demo_meals[:-1], constraint_results=[], explanations=[], generated_at='now'))  # noqa: E501
expect_error(lambda: require_authoritative_weekly_solver(), NotImplementedError)
expect_error(lambda: store.save(swapped))
try:
    store.get(demo_plan.plan_id, 'different-user')
except KeyError:
    pass
else:
    raise AssertionError('Owner isolation failed')
assert len({(m.day, m.slot) for m in swapped.meals}) == 21
assert swapped.validated_by_csp is False
print('PASS: schema, ownership, release validation, 21-slot uniqueness, revisions, swaps and solver guard')  # noqa: E501


PASS: schema, ownership, release validation, 21-slot uniqueness, revisions, swaps and solver guard


## 11. Remaining work before production integration

1. Agree final profile fields and canonical mappings with your teammate; align these proposals with actual UI/DB fields.
2. Implement authenticated backend routes and durable profile/plan repositories; the uploaded backend currently only provides a skeleton.
3. Implement trusted KB and source snapshot selection with exact version/digest validation; do not substitute the demo registry.
4. Build a **real** profile-to-`RuleContext` adapter using canonical, evidence-backed facts and release bindings.
5. Connect the real candidate/combination/daily reasoning DTOs to your data pipeline, with valid approved KB snapshots.
6. Implement and test the missing weekly CSP coordinator and week-level constraints before claiming a validated 21-meal plan.
7. Re-run affected recipe, day and week constraints for swaps/regeneration; preserve provenance and explanation traces.
8. Add API integration tests, real fixture data, DB migrations, authentication and CI validation.

**Notebook execution guarantee:** Local demonstration and contract smoke tests run without the repository or datasets. Real AI execution is conditional on installed dependencies, approved snapshots, actual candidate facts and the missing weekly coordinator; no guarantee of error-free production integration is possible from the supplied files alone.

# Part II — Qwen, NLU, entity normalization, dialogue and conversational UI

This section preserves the user's original conversational assistant, with portable dataset discovery and optional Qwen/interactive UI. It is a **separate conversational prototype**, not a wired FastAPI endpoint. Install `pandas` for recipe data. Qwen requires `transformers`, `torch`, and `accelerate`; Gradio/ipywidgets are optional. A dataset-free NLU smoke test is included, so the notebook can still execute without `RAW_recipes.csv`. Hard allergy enforcement and validated 21-meal planning must remain delegated to authoritative reasoning services.

In [10]:
# Optional dependency setup: uncomment in VS Code/Jupyter when needed.
# %pip install pandas numpy rapidfuzz
# %pip install transformers torch accelerate sentencepiece
# %pip install gradio ipywidgets


In [11]:
# Savora: notebook-friendly Food.com conversational recipe assistant.
# Qwen is optional; rule-based fallback works without a GPU or internet.
import ast
import copy
import json
import re
from dataclasses import dataclass, field

import pandas as pd

ALIASES = {'potatoes': 'potato', 'tomatoes': 'tomato', 'onions': 'onion', 'chillies': 'chili', 'chilies': 'chili', 'capsicum': 'bell pepper', 'bell peppers': 'bell pepper', 'brinjal': 'eggplant', 'aubergine': 'eggplant', 'garbanzo beans': 'chickpeas', 'chickpea': 'chickpeas', 'coriander leaves': 'cilantro', 'curd': 'yogurt'}  # noqa: E501
DIET_ALIASES = {'gluten free': 'gluten-free', 'dairy free': 'dairy-free', 'low carb': 'low-carb', 'low fat': 'low-fat', 'low sodium': 'low-sodium', 'plant based': 'vegan'}  # noqa: E501
CUISINES = {'indian', 'italian', 'mexican', 'chinese', 'thai', 'japanese', 'french', 'greek', 'korean', 'mediterranean', 'american', 'vietnamese'}  # noqa: E501
DIETS = {'vegetarian', 'vegan', 'gluten-free', 'dairy-free', 'low-carb', 'low-fat', 'low-sodium'}

def normalize(text):
    text = re.sub('\\s+', ' ', str(text).lower().replace('_', ' ').strip())
    text = re.sub('[^\\w\\s-]', '', text)
    return ALIASES.get(text, DIET_ALIASES.get(text, text))

def parse_list(value):
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []
    try:
        out = ast.literal_eval(value)
        return out if isinstance(out, list) else []
    except (SyntaxError, ValueError, TypeError, MemoryError):
        return []

@dataclass
class Slots:
    ingredients: list = field(default_factory=list)
    excluded: list = field(default_factory=list)
    cuisine: str | None = None
    diet: str | None = None
    max_minutes: int | None = None

class RecipeStore:

    def __init__(self, path, max_rows=None):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f'{path} not found. Set DATA_DIR to your CSV folder.')
        cols = ['id', 'name', 'minutes', 'ingredients', 'tags', 'steps', 'description']
        self.df = pd.read_csv(path, usecols=cols, nrows=max_rows, low_memory=False)
        self.df['minutes'] = pd.to_numeric(self.df['minutes'], errors='coerce')
        self.df['name'] = self.df['name'].fillna('Untitled recipe')
        for col in ('ingredients', 'tags', 'steps'):
            self.df[col] = self.df[col].map(parse_list)
        self.df['ingredient_set'] = self.df['ingredients'].map(lambda values: {normalize(v) for v in values})  # noqa: E501
        self.df['tag_set'] = self.df['tags'].map(lambda values: {normalize(v) for v in values})
        self.vocabulary = set().union(*self.df['ingredient_set'].tolist()) if len(self.df) else set()  # noqa: E501
        self.vocabulary.update(ALIASES)

    def search(self, slots, limit=5):
        df = self.df
        if slots.max_minutes is not None:
            df = df[df['minutes'].le(slots.max_minutes)]
        if slots.cuisine:
            df = df.loc[df['tag_set'].map(lambda x: slots.cuisine in x).astype(bool)]
        if slots.diet:
            df = df.loc[df['tag_set'].map(lambda x: slots.diet in x).astype(bool)]
        if slots.ingredients:
            df = df.loc[df['ingredient_set'].map(lambda x: all(v in x for v in slots.ingredients)).astype(bool)]  # noqa: E501
        if slots.excluded:
            df = df.loc[df['ingredient_set'].map(lambda x: not any(v in x for v in slots.excluded)).astype(bool)]  # noqa: E501
        if df.empty:
            return []
        if df.empty:
            return []
        return df.sort_values(['minutes', 'id']).head(limit).to_dict('records')

class NLU:

    def __init__(self, vocabulary, use_qwen=False, model_id='Qwen/Qwen2.5-1.5B-Instruct'):
        self.vocabulary = vocabulary
        self.pipe = None
        if use_qwen:
            try:
                from transformers import pipeline
                kwargs = {'model': model_id, 'device_map': 'auto', 'torch_dtype': 'auto'}
                self.pipe = pipeline('text-generation', **kwargs)
            except Exception as exc:
                raise RuntimeError('Could not load Qwen. Install torch, transformers and accelerate; or use use_qwen=False. Original error: ' + str(exc)) from exc  # noqa: E501

    def _rules(self, utterance):
        text = normalize(utterance)
        if re.search('\\b(reset|start over|clear preferences)\\b', text):
            return {'intent': 'reset'}
        if re.search('\\b(help|what can you do)\\b', text):
            return {'intent': 'help'}
        cuisine = next((x for x in sorted(CUISINES, key=len, reverse=True) if re.search('(?<!\\w)' + re.escape(x) + '(?!\\w)', text)), None)  # noqa: E501
        diet = next((x for x in sorted(DIETS, key=len, reverse=True) if re.search('(?<!\\w)' + re.escape(x) + '(?!\\w)', text)), None)  # noqa: E501
        for alias, value in DIET_ALIASES.items():
            if re.search('(?<!\\w)' + re.escape(alias) + '(?!\\w)', text):
                diet = value
        m = re.search('\\b(?:under|less than|within|max(?:imum)?|up to|in)\\s*(\\d+)\\s*(?:minutes?|mins?)\\b', text)  # noqa: E501
        minutes = int(m.group(1)) if m else None
        positive, excluded = ([], [])
        for term in sorted(self.vocabulary, key=len, reverse=True):
            if len(term) < 3 or not re.search('(?<!\\w)' + re.escape(term) + '(?!\\w)', text):
                continue
            canonical = normalize(term)
            if re.search('\\b(?:without|exclude|remove|no|avoid)\\s+(?:any\\s+)?' + re.escape(term) + '(?!\\w)', text):  # noqa: E501
                if canonical not in excluded:
                    excluded.append(canonical)
            elif canonical not in positive:
                positive.append(canonical)
        positive = [x for x in positive if not any(x != y and x in y for y in positive)]
        return {'intent': 'search', 'ingredients': positive[:10], 'excluded': excluded[:10], 'cuisine': cuisine, 'diet': diet, 'max_minutes': minutes}  # noqa: E501

    def extract(self, utterance):
        fallback = self._rules(utterance)
        if self.pipe is None or fallback.get('intent') in ('reset', 'help'):
            return fallback
        instruction = 'Extract recipe preferences from the user message. Return ONLY JSON with keys intent, ingredients, excluded, cuisine, diet, max_minutes. Use null for absent scalar values and [] for absent arrays. Intent is search, reset or help. Do not invent preferences.'  # noqa: E501
        try:
            messages = [{'role': 'system', 'content': instruction}, {'role': 'user', 'content': utterance}]  # noqa: E501
            result = self.pipe(messages, max_new_tokens=180, do_sample=False)
            generated = result[0]['generated_text']
            answer = generated[-1]['content'] if isinstance(generated, list) else generated
            match = re.search('\\{.*?\\}', answer, flags=re.S)
            obj = json.loads(match.group()) if match else {}
            clean = {'intent': obj.get('intent', 'search')}
            for key in ('ingredients', 'excluded'):
                values = obj.get(key) or []
                if isinstance(values, str):
                    values = [values]
                clean[key] = list(dict.fromkeys(normalize(x) for x in values if isinstance(x, str) and normalize(x) in self.vocabulary))[:10]  # noqa: E501
            for key, allowed in (('cuisine', CUISINES), ('diet', DIETS)):
                value = normalize(obj[key]) if isinstance(obj.get(key), str) else None
                clean[key] = value if value in allowed else None
            try:
                val = obj.get('max_minutes')
                clean['max_minutes'] = int(val) if val is not None and 0 < int(val) < 10000 else None  # noqa: E501
            except (ValueError, TypeError):
                clean['max_minutes'] = None
            for key in ('ingredients', 'excluded', 'cuisine', 'diet', 'max_minutes'):
                if not clean.get(key):
                    clean[key] = fallback.get(key)
            return clean
        except (ValueError, TypeError, KeyError, IndexError, AttributeError):
            return fallback

class SavoraBot:

    def __init__(self, data_dir='.', use_qwen=False, model_id='Qwen/Qwen2.5-1.5B-Instruct', max_rows=None):  # noqa: E501
        self.store = RecipeStore(Path(data_dir) / 'RAW_recipes.csv', max_rows=max_rows)
        self.nlu = NLU(self.store.vocabulary, use_qwen=use_qwen, model_id=model_id)
        self.state = Slots()
        self.last_results = []

    def reset(self):
        self.state = Slots()
        self.last_results = []

    def respond(self, message):
        if not str(message).strip():
            return 'Please describe what you would like to cook.'
        obj = self.nlu.extract(message)
        if obj.get('intent') == 'reset':
            self.reset()
            return 'Preferences cleared. What would you like to cook?'
        if obj.get('intent') == 'help':
            return 'Try: "Vegetarian Indian recipe with potato under 30 minutes". Follow up with "under 20 minutes", "without onion", or "reset".'  # noqa: E501
        for key in ('cuisine', 'diet', 'max_minutes'):
            if obj.get(key) is not None:
                setattr(self.state, key, obj[key])
        for item in obj.get('excluded', []):
            if item not in self.state.excluded:
                self.state.excluded.append(item)
            if item in self.state.ingredients:
                self.state.ingredients.remove(item)
        for item in obj.get('ingredients', []):
            if item not in self.state.ingredients:
                self.state.ingredients.append(item)
            if item in self.state.excluded:
                self.state.excluded.remove(item)
        if not any((self.state.ingredients, self.state.excluded, self.state.cuisine, self.state.diet, self.state.max_minutes)):  # noqa: E501
            return 'What ingredients, cuisine, diet, or cooking time would you prefer?'
        self.last_results = self.store.search(self.state)
        if not self.last_results:
            return 'No exact matches for these preferences. Try a longer cooking time, fewer required ingredients, or say "reset".'  # noqa: E501
        lines = ['Here are recipes matching your current preferences:']
        for index, rec in enumerate(self.last_results, 1):
            mins = f"{rec['minutes']:g}" if pd.notna(rec['minutes']) else '?'
            lines.append(f"{index}. **{str(rec['name']).title()}** — {mins} min (recipe ID {rec['id']})")  # noqa: E501
            lines.append('   Ingredients: ' + ', '.join(map(str, rec['ingredients'][:8])))
        lines.append('Refine your search with a follow-up, or type "reset".')
        return '\n'.join(lines)

    def recipe_details(self, result_number=1):
        if not 1 <= result_number <= len(self.last_results):
            return 'Choose a recipe number from your most recent results.'
        rec = self.last_results[result_number - 1]
        steps = '\n'.join((f'{i}. {step}' for i, step in enumerate(rec['steps'], 1)))
        return f"{rec['name'].title()}\nIngredients: {', '.join(rec['ingredients'])}\nInstructions:\n{steps}"  # noqa: E501

def launch_gradio(bot, share=False):
    """Launch a local prototype with independent state reconstructed from history.

    This interface is for local demonstration, not production authentication or
    allergy-safety validation. The backend reasoning engine must enforce safety.
    """
    import gradio as gr

    def reply(message, history):
        session_bot = copy.copy(bot)
        session_bot.state = Slots()
        session_bot.last_results = []
        for item in history or []:
            if isinstance(item, dict) and item.get('role') == 'user':
                session_bot.respond(str(item.get('content', '')))
            elif isinstance(item, (list, tuple)) and item:
                session_bot.respond(str(item[0]))
        return str(session_bot.respond(message))
    demo = gr.ChatInterface(fn=reply, title='Savora AI Meal Planner', description='Ask Savora for recipe recommendations (local prototype).')  # noqa: E501
    return demo.launch(share=share, inbrowser=True)


## Conversational configuration and portable dataset discovery

Searches the repository and common dataset locations without assuming any person's Windows username. If `RAW_recipes.csv` is unavailable, NLU and normalization checks still run. A large Food.com dataset is **not** bundled with this notebook.

In [12]:
import os

USE_QWEN = False
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_ROWS = 10000

def locate_recipe_dataset():
    explicit = os.environ.get('SAVORA_DATA_DIR', '')
    roots = [Path(explicit)] if explicit else []
    repo = find_repo_root()
    if repo is not None:
        roots += [repo / 'Food.com', repo / 'Data' / 'Food.com', repo / 'data' / 'Food.com', repo]
    roots += [Path.cwd(), Path.cwd() / 'Food.com', Path.cwd().parent / 'Food.com', Path.cwd() / 'Data' / 'Food.com', Path.home() / 'Downloads' / 'Food.com', Path.home() / 'Downloads', Path.home() / 'Documents']  # noqa: E501
    for root in roots:
        candidate = root / 'RAW_recipes.csv'
        if candidate.is_file():
            return candidate
    return None
RECIPE_CSV = locate_recipe_dataset()
print('Recipe dataset:', RECIPE_CSV or 'not found — conversational dataset tests will be skipped')


Recipe dataset: c:\Users\Aryaman\OneDrive\Desktop\MR2 Workshop\Savora-AI-Meal-Planner-GC-1-1\Food.com\RAW_recipes.csv


In [13]:
# Run without the Food.com dataset: entity normalization and rule-based NLU.
assert normalize('brinjal') == 'eggplant'
assert normalize('capsicum') == 'bell pepper'
standalone_nlu = NLU({'potato', 'eggplant', 'bell pepper'}, use_qwen=False)
print('NLU smoke test:', standalone_nlu.extract('I want a vegetarian Indian potato dish under 30 minutes'))  # noqa: E501
print('Entity normalization smoke tests passed')


NLU smoke test: {'intent': 'search', 'ingredients': ['potato'], 'excluded': [], 'cuisine': 'indian', 'diet': 'vegetarian', 'max_minutes': 30}
Entity normalization smoke tests passed


In [14]:
# Full chatbot requires the original Food.com RAW_recipes.csv.
chatbot = None
if RECIPE_CSV is not None:
    chatbot = SavoraBot(data_dir=RECIPE_CSV.parent, use_qwen=USE_QWEN, model_id=MODEL_ID, max_rows=MAX_ROWS)  # noqa: E501
    print("Recipes loaded:", len(chatbot.store.df))
    print("Chatbot example:", chatbot.respond("vegetarian Indian dish with potato under 30 minutes"))  # noqa: E501
else:
    print("Skipped dataset-backed chatbot. Set SAVORA_DATA_DIR to the "
        "folder containing RAW_recipes.csv and rerun.")


Recipes loaded: 10000
Chatbot example: Here are recipes matching your current preferences:
1. **Tide Me Over   Indian Chaat  Simple Veggie Salad** — 2 min (recipe ID 63793)
   Ingredients: cucumber, potato, chopped tomato, plain yogurt, chat masala, paprika, salt
2. **Aloo  Potato  Pakora** — 10 min (recipe ID 110215)
   Ingredients: potatoes, corn oil, besan, water, red chili powder, cumin powder, garlic, ginger
3. **Aloo Curry  Potato Curry** — 25 min (recipe ID 11512)
   Ingredients: potatoes, onions, garlic, ginger, tomatoes, green chilies, salt, turmeric
4. **Aaloo Chaat  Tangy Potatoes** — 25 min (recipe ID 96178)
   Ingredients: potato, red chili powder, chat masala, coriander powder, onion, salt, tamarind pulp, oil
5. **Aloo Baigain  Potato And Eggplant  Aubergine** — 25 min (recipe ID 149969)
   Ingredients: potato, eggplants, red chili powder, cumin, coriander powder, turmeric powder, salt, garlic paste
Refine your search with a follow-up, or type "reset".


## Optional interactive chat

These cells are opt-in and require `ipywidgets` or `gradio` installed in the active notebook kernel. They are intentionally not auto-launched.

In [15]:
# Optional: launch a browser UI (uncomment when dataset and Gradio are available).
# if chatbot is not None:
#     launch_gradio(chatbot)

# Optional: run Qwen-backed NLU (downloads model weights; opt in explicitly).
# if RECIPE_CSV is not None:
#     qwen_chatbot = SavoraBot(
#         data_dir=RECIPE_CSV.parent, use_qwen=True,
#         model_id=MODEL_ID, max_rows=MAX_ROWS,
#     )
#     print(qwen_chatbot.nlu.extract("Find vegetarian Indian recipes without onion"))


## Integration boundary and limitations

- The original `SavoraBot` and its `Slots` are conversational state, **not** the validated `UserProfile` or authoritative `RuleContext` from the reasoning package.
- Convert normalized entities and slot values to a versioned, validated profile through a dedicated backend adapter; never silently reinterpret unknown allergens as safe.
- Only call `filter_candidates`, `evaluate_combination`, and `solve_daily_plan` with real validated snapshot DTOs and accepted KB/source provenance.
- The repository snapshot does **not** provide a validated 7-day CSP optimizer or fully implemented meal-plan backend routes. Contract placeholders must not be presented as actual recommendations.
- `MAX_ROWS` is a development performance setting; using a partial dataset changes retrieval coverage. Qwen is opt-in, not necessary for the rule-based smoke tests.
- To avoid disrupting your teammate's work, keep this notebook in a separate folder and integrate through a pull request after interface agreement.

# Part III — Eleven additional responsibilities: implementation audit

| # | Required work | Repository snapshot / notebook status | Deliverable here |
|---|---|---|---|
| 1 | Joint profile schema, canonical allergen IDs, UNKNOWN not safe | Previous notebook has a draft profile; canonical IDs and approval not established | Strict reference profile + fail-closed eligibility tests |
| 2 | SQLAlchemy models + Alembic migrations (Q6 combination tables) | Backend `alembic/versions` has only `.gitkeep` | Proposed table inventory and migration checklist; **not migrated** |
| 3 | Auth, onboarding, versioned nutrition target calculation | Backend has only `/api/health` | Proposed contracts and snapshot checks; **not deployed** |
| 4 | Profile → `RuleContext` with accepted KB | Real reasoning package exists; accepted production release not supplied | Fail-closed adapter boundary; **requires approved KB** |
| 5 | Seed-dish shortlist/save/rank and torch scorer | Team approval pending; ranking implementation absent in snapshot | Gated endpoint specifications; **not implemented** |
| 6 | Candidates → rules → composition → rank → weekly CSP → save → explain | Daily reasoning exists; weekly CSP not present | Fail-closed orchestrator; **not a 21-meal solver** |
| 7 | Qwen `parse_craving` → retrieval → rules → rank (no composer/CSP) | Notebook has optional Qwen + rules baseline, no accepted typed craving API | Validated parse schema and reference craving router |
| 8 | Swap locked slots + before/after snapshot | Local SQLite draft only | Revision and locked-slot contract tests; **not production storage** |
| 9 | Next.js screens + OpenAPI types | Frontend is scaffold; `packages/contracts` exports empty object | UI and contract requirements; **not built by notebook** |
| 10 | API integration + E2E evaluation | Backend health test only in supplied snapshot | Offline contract tests and E2E checklist |
| 11 | NLU baseline vs Qwen: intent accuracy, slot F1, schema validity, latency | Smoke tests only | Reproducible evaluation harness; Qwen metrics require running model |

**Non-negotiable:** No generative output or unknown ingredient/allergen can be marked safe. Real rule engine is authoritative. This notebook cannot assert accepted KB identities without team-approved evidence.

## 12. Canonical IDs, profile and craving parse schemas (reference contracts)

IDs below are examples of **format**, not an accepted food taxonomy. The registry must be loaded from the team's approved source. Unrecognized allergens are flagged for manual resolution, never silently accepted as safe. This notebook uses the Python standard library for contract checks so the validation tests run without a database or Qwen model.

In [16]:
import statistics
import time

CANONICAL_ALLERGEN_IDS = frozenset({'allergen:peanut', 'allergen:milk', 'allergen:egg', 'allergen:soy', 'allergen:wheat', 'allergen:fish', 'allergen:shellfish', 'allergen:tree_nut', 'allergen:sesame'})  # noqa: E501
ALLERGEN_ALIASES = {'peanut': 'allergen:peanut', 'peanuts': 'allergen:peanut', 'dairy': 'allergen:milk', 'milk': 'allergen:milk', 'eggs': 'allergen:egg', 'egg': 'allergen:egg', 'soy': 'allergen:soy', 'sesame': 'allergen:sesame'}  # noqa: E501

def canonicalize_allergens(raw_values, registry=CANONICAL_ALLERGEN_IDS):
    ids, unknown = ([], [])
    for raw in raw_values:
        token = str(raw).strip().lower()
        canonical = ALLERGEN_ALIASES.get(token, token)
        if canonical in registry:
            ids.append(canonical)
        else:
            unknown.append(raw)
    return (sorted(set(ids)), sorted(set(map(str, unknown))))

@dataclass
class SafeProfileDraft:
    user_id: str
    diet_pattern: str
    allergy_ids: list[str]
    unresolved_allergies: list[str] = field(default_factory=list)
    explicit_exclusions: list[str] = field(default_factory=list)
    conditions: list[str] = field(default_factory=list)
    cuisine_policy: str = 'mixed'
    preferred_cuisines: list[str] = field(default_factory=list)
    age_years: int | None = None
    height_cm: float | None = None
    weight_kg: float | None = None
    activity_level: str | None = None
    nutrition_goal: str | None = None
    target_snapshot_version: int | None = None

    def validate(self):
        if not self.user_id.strip():
            raise ValueError('user_id required')
        if self.cuisine_policy not in ('strict', 'majority', 'mixed'):
            raise ValueError('invalid cuisine policy')
        if any(a not in CANONICAL_ALLERGEN_IDS for a in self.allergy_ids):
            raise ValueError('unknown canonical allergen ID')
        if self.unresolved_allergies:
            raise ValueError('Unresolved allergens: cannot authorize safe meal planning')
        if self.age_years is not None and (not 1 <= self.age_years <= 120):
            raise ValueError('age out of range')
        if self.weight_kg is not None and self.weight_kg <= 0:
            raise ValueError('invalid weight')
        return self

@dataclass
class CravingParse:
    intent: str
    ingredients: list[str] = field(default_factory=list)
    excluded_ingredients: list[str] = field(default_factory=list)
    cuisine: str | None = None
    diet: str | None = None
    max_minutes: int | None = None
    confidence: float | None = None

    @classmethod
    def from_mapping(cls, payload):
        if not isinstance(payload, dict):
            raise ValueError('Qwen output must be a JSON object')
        allowed = set(cls.__dataclass_fields__)
        if set(payload) - allowed:
            raise ValueError('Unknown craving fields')
        obj = cls(**payload)
        if obj.intent not in {'search', 'refine', 'reset', 'help'}:
            raise ValueError('Invalid intent')
        for key in ('ingredients', 'excluded_ingredients'):
            values = getattr(obj, key)
            if not isinstance(values, list) or any(not isinstance(v, str) for v in values):
                raise ValueError(key + ' must be list[str]')
        if obj.max_minutes is not None and (type(obj.max_minutes) is not int or not 1 <= obj.max_minutes <= 1440):  # noqa: E501
            raise ValueError('Invalid max_minutes')
        if obj.confidence is not None and (not 0 <= obj.confidence <= 1):
            raise ValueError('Invalid confidence')
        return obj
profile_example = SafeProfileDraft(user_id='demo-user', diet_pattern='vegetarian', allergy_ids=['allergen:peanut'], age_years=24)  # noqa: E501
print('Reference profile:', asdict(profile_example.validate()))
print('Unrecognized allergens (must be resolved):', canonicalize_allergens(['peanuts', 'unmapped-food']))  # noqa: E501


Reference profile: {'user_id': 'demo-user', 'diet_pattern': 'vegetarian', 'allergy_ids': ['allergen:peanut'], 'unresolved_allergies': [], 'explicit_exclusions': [], 'conditions': [], 'cuisine_policy': 'mixed', 'preferred_cuisines': [], 'age_years': 24, 'height_cm': None, 'weight_kg': None, 'activity_level': None, 'nutrition_goal': None, 'target_snapshot_version': None}
Unrecognized allergens (must be resolved): (['allergen:peanut'], ['unmapped-food'])


## 13. Safety boundary: approved KB + RuleContext

`UNKNOWN` is **not** equivalent to `PASS`. The application must resolve all ingredient derivatives and provenance through an accepted knowledge snapshot before asserting eligibility. The adapter below refuses unapproved releases and unresolved allergies. It intentionally does not fabricate the repository's `RuleContext` constructor arguments.

In [17]:
@dataclass(frozen=True)
class AcceptedKB:
    kb_id: str
    version: str
    digest: str
    source_ids: tuple[str, ...]
    accepted_by: str

    def validate(self):
        if not all([self.kb_id, self.version, self.digest, self.source_ids, self.accepted_by]):
            raise ValueError('Incomplete accepted KB identity')
        if not re.fullmatch('[0-9a-f]{64}', self.digest):
            raise ValueError('Expected SHA-256 digest')
        return self

def profile_to_rule_context(profile, accepted_kb=None, actual_factory=None):
    profile.validate()
    if accepted_kb is None:
        raise RuntimeError('No team-accepted KB snapshot: fail closed')
    accepted_kb.validate()
    if actual_factory is None:
        raise NotImplementedError('Wire approved savora_ai RuleContext factory here')
    return actual_factory(profile, accepted_kb)

def allergen_eligibility(recipe_allergen_ids, profile, provenance_complete=False):
    profile.validate()
    if not provenance_complete or recipe_allergen_ids is None:
        return 'UNKNOWN'
    if not set(recipe_allergen_ids).issubset(CANONICAL_ALLERGEN_IDS):
        return 'UNKNOWN'
    return 'FAIL' if set(recipe_allergen_ids) & set(profile.allergy_ids) else 'PASS'
assert allergen_eligibility(None, profile_example) == 'UNKNOWN'
assert allergen_eligibility(['allergen:peanut'], profile_example, True) == 'FAIL'
assert allergen_eligibility([], profile_example, True) == 'PASS'
print('Fail-closed allergy checks passed')


Fail-closed allergy checks passed


## 14. Backend database, authentication, nutrition targets and onboarding

**Implementation belongs in** `apps/backend/src/savora_backend/` and `apps/backend/alembic/versions/` using async SQLAlchemy and Alembic, not in a notebook.

Proposed database tables: `users`, `profiles`, `nutrition_target_snapshots`, `accepted_kb_releases`, `plans`, `plan_slots`, `plan_revisions`, `substitutions`, `meal_combinations`, `meal_combination_components`, `seed_dish_picks`. Final fields, foreign keys, unique constraints, indexes and deletion policy require agreement under **contract Q6**.

Proposed API contracts (not existing endpoints):

| Route | Auth | Input | Response / effect |
|---|---|---|---|
| `POST /api/v1/onboarding` | Required | Profile input + explicit safety confirmations | Validated profile + versioned nutrition target snapshot |
| `PUT /api/v1/profiles/me` | Required | Profile update + expected version | New profile version and recomputed targets if relevant fields change |
| `GET /api/v1/profiles/me` | Required | None | Current profile + target snapshot version |
| `POST /api/v1/seed-dishes/shortlist` | Required | Accepted KB + profile version | Rule-filtered and ranked shortlist |
| `POST /api/v1/seed-dishes/picks` | Required | Chosen recipe IDs | Persist seed picks and their provenance |
| `POST /api/v1/meal-plans` | Required | Profile snapshot, seed picks, KB/model versions | `202 job_id` until verified weekly CSP completes |
| `POST /api/v1/cravings/search` | Required | User utterance | Schema-validated parse + rule-eligible ranked results |
| `POST /api/v1/meal-plans/{id}/swaps` | Required | Slot, candidate, revision, locked slots | Verified new revision with before/after snapshot |

**Nutrition calculation:** call the team's approved `calculate_nutrition_targets` implementation. The provided repository snapshot does not establish this function. Never invent clinical thresholds or claim an estimated target is clinically validated.

In [18]:
@dataclass(frozen=True)
class TargetSnapshot:
    user_id: str
    profile_version: int
    target_version: int
    calculation_method: str
    source_id: str
    kcal: float
    protein_g: float

    def validate(self):
        if min(self.profile_version, self.target_version) < 1:
            raise ValueError('version must be positive')
        if self.kcal <= 0 or self.protein_g <= 0:
            raise ValueError('nutrition targets must be positive')
        if not self.calculation_method or not self.source_id:
            raise ValueError('provenance required')
        return self

def onboarding_reference(profile, calculate_nutrition_targets=None):
    profile.validate()
    if calculate_nutrition_targets is None:
        raise NotImplementedError('Approved calculate_nutrition_targets is required for onboarding')
    target = calculate_nutrition_targets(profile)
    if not isinstance(target, TargetSnapshot):
        raise TypeError('Expected versioned TargetSnapshot')
    return {'profile': asdict(profile), 'targets': asdict(target.validate())}
try:
    onboarding_reference(profile_example)
except NotImplementedError:
    print('Onboarding correctly blocked: approved nutrition calculator not connected')


Onboarding correctly blocked: approved nutrition calculator not connected


## 15. Seed-dish approval gate, scorer and full plan pipeline

**Do not activate seed-dish endpoints until the team approves the step.** When approved, shortlist = rule filter → `rank_recipes` (empty history); selected picks are stored and supplied as history to `rank_combinations`. Torch dependency, model weights, model version, scorer loading and inference ownership must be agreed. The notebook does not download or deliver the team's ranking model.

**Plan pipeline:** candidate retrieval → authoritative rule eligibility → composition → ranking → **weekly CSP (21 slots)** → persistence → explanation. The current repository snapshot has a daily solver but no production weekly solver. Never save 21 placeholders as a valid plan.

In [19]:
def seed_dish_shortlist(profile, team_approved=False, rule_filter=None, rank_recipes=None, recipes=None):  # noqa: E501
    profile.validate()
    if not team_approved:
        raise PermissionError('Seed-dish step requires team approval')
    if rule_filter is None or rank_recipes is None:
        raise NotImplementedError('Rule engine and ranking scorer required')
    eligible = rule_filter(recipes, profile)
    return rank_recipes(eligible, history=[])

def plan_pipeline(*, profile, kb, candidates=None, rule_filter=None, compose=None, rank_combinations=None, weekly_solver=None, persist=None, explain=None):  # noqa: E501
    profile.validate()
    kb.validate()
    for name, fn in [('candidate retrieval', candidates), ('rules', rule_filter), ('composition', compose), ('ranking', rank_combinations), ('weekly CSP', weekly_solver), ('persistence', persist), ('explanation', explain)]:  # noqa: E501
        if fn is None:
            raise NotImplementedError(name + ' implementation required')
    pool = candidates(profile, kb)
    eligible = rule_filter(pool, profile, kb)
    combos = compose(eligible, profile)
    ranked = rank_combinations(combos, profile)
    solution = weekly_solver(ranked, profile)
    if len(solution.get('slots', [])) != 21 or not solution.get('all_hard_constraints_passed', False):  # noqa: E501
        raise ValueError('Weekly solution unverified: cannot persist')
    saved = persist(solution, profile, kb)
    return {'plan': saved, 'explanations': explain(saved)}
try:
    seed_dish_shortlist(profile_example)
except PermissionError:
    print('Seed-dish gate correctly blocked')


Seed-dish gate correctly blocked


## 16. Craving flow: typed Qwen parse → retrieval → rules → ranking

No meal composer and **no CSP** on this standalone craving path. Qwen extracts intent/slots only; it must **never** make safety decisions. Actual retrieval, authoritative rules and trained ranker must be connected by the backend. Missing safety evidence yields UNKNOWN, and is not shown as safe.

In [20]:
def parse_craving_json(model_text):
    return CravingParse.from_mapping(json.loads(model_text))

def craving_pipeline(utterance, *, parser, retrieval, rules, ranker, profile, kb):
    profile.validate()
    kb.validate()
    parsed = parser(utterance)
    if not isinstance(parsed, CravingParse):
        raise TypeError('Parser must return validated CravingParse')
    candidates = retrieval(parsed)
    decisions = rules(candidates, profile, kb)
    eligible = [recipe for recipe, status in decisions if status == 'PASS']
    return {'parsed': asdict(parsed), 'recipes': ranker(eligible), 'excluded_unknown_or_failed': len(decisions) - len(eligible)}  # noqa: E501
parsed_example = parse_craving_json('{"intent":"search","ingredients":["potato"],"cuisine":"indian","diet":"vegetarian","max_minutes":30}')  # noqa: E501
assert parsed_example.max_minutes == 30
print('Schema-validated craving parse:', asdict(parsed_example))


Schema-validated craving parse: {'intent': 'search', 'ingredients': ['potato'], 'excluded_ingredients': [], 'cuisine': 'indian', 'diet': 'vegetarian', 'max_minutes': 30, 'confidence': None}


## 17. Swaps, locked slots, snapshots and concurrency

Proposed API must validate plan ownership, optimistic revision, locked slots, nutrition/constraints and source versions. A substitution creates a new immutable revision with **before** and **after** snapshots. Do not overwrite the old plan or silently keep stale PASS statuses.

In [21]:
def propose_swap(plan, *, slot, replacement, expected_revision, locked_slots=()):
    if plan['revision'] != expected_revision:
        raise RuntimeError('409 revision conflict')
    if slot in locked_slots:
        raise PermissionError('Locked slot cannot be swapped')
    if slot not in plan['slots']:
        raise KeyError('Unknown slot')
    before = copy.deepcopy(plan)
    after = copy.deepcopy(plan)
    after['slots'][slot] = replacement
    after['revision'] += 1
    after['constraint_status'] = 'PENDING_REVALIDATION'
    return {'before': before, 'after': after, 'status': 'PENDING_SOLVER', 'persisted': False}
sample_plan = {'plan_id': 'demo', 'revision': 1, 'slots': {'D1_LUNCH': 'recipe-a'}, 'constraint_status': 'PASS'}  # noqa: E501
proposal = propose_swap(sample_plan, slot='D1_LUNCH', replacement='recipe-b', expected_revision=1)
assert proposal['before']['slots']['D1_LUNCH'] == 'recipe-a'
assert proposal['after']['constraint_status'] == 'PENDING_REVALIDATION'
assert sample_plan['revision'] == 1
print('Swap preserves before/after; new plan not falsely marked valid')


Swap preserves before/after; new plan not falsely marked valid


## 18. Frontend and OpenAPI contracts — integration checklist

A Jupyter notebook **cannot create a working Next.js application by itself**. Required implementation files and ownership:

- `apps/frontend/src/core/private/UserProfile/`: onboarding form, explicit allergy confirmations, versioned target summary.
- `apps/frontend/src/core/private/MealPlanning/`: 21-slot schedule, constraint explanations, locked slots and swap UI.
- `apps/frontend/src/core/private/Craving/`: free-text craving search, uncertainty and safety messaging.
- `apps/backend/src/savora_backend/`: real authenticated routes, services, ORM, Alembic migration, idempotency, authorization and validation.
- `packages/contracts/`: generate TypeScript API types from FastAPI OpenAPI schema, ensure request/response agreement.

**Repository audit:** `apps/frontend/src/app/page.tsx` is a scaffold and `packages/contracts/src/index.ts` exports `{}` in the uploaded snapshot. None of these screens can be claimed complete from the notebook.

## 19. NLU evaluation: template baseline vs Qwen

Evaluation metrics: intent accuracy, micro-averaged slot precision/recall/F1, schema validity rate and latency (p50/p95). Gold cases below are tiny illustrative fixtures, **not** a valid project benchmark. To evaluate Qwen, pass a real model parser; never report Qwen scores if it was not run. Report dataset size, hardware, model ID, quantization, decoding settings and prompt version.

In [22]:
def evaluate_nlu(examples, predictor):
    times = []
    correct = 0
    tp = fp = fn = valid = 0
    errors = []
    for item in examples:
        t0 = time.perf_counter()
        try:
            pred = predictor(item['text'])
            if isinstance(pred, dict):
                pred = CravingParse.from_mapping(pred)
            if not isinstance(pred, CravingParse):
                raise TypeError('Expected CravingParse')
            valid += 1
            correct += int(pred.intent == item['gold'].intent)

            def pairs(x):
                d = asdict(x)
                d.pop('confidence', None)
                d.pop('intent', None)
                return {(k, str(v)) for k, val in d.items() for v in (val if isinstance(val, list) else [val]) if v is not None}  # noqa: E501
            p, g = (pairs(pred), pairs(item['gold']))
            tp += len(p & g)
            fp += len(p - g)
            fn += len(g - p)
        except Exception as exc:
            errors.append(type(exc).__name__ + ': ' + str(exc))
        times.append((time.perf_counter() - t0) * 1000)
    n = len(examples)
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    return {'samples': n, 'intent_accuracy': correct / n if n else 0, 'slot_precision': precision, 'slot_recall': recall, 'slot_f1': 2 * precision * recall / (precision + recall) if precision + recall else 0, 'schema_validity': valid / n if n else 0, 'latency_p50_ms': statistics.median(times) if times else None, 'latency_p95_ms': sorted(times)[min(n - 1, int(0.95 * n))] if n else None, 'errors': errors}  # noqa: E501
EVAL_FIXTURES = [{'text': 'vegetarian Indian potato under 30 minutes', 'gold': CravingParse(intent='search', ingredients=['potato'], cuisine='indian', diet='vegetarian', max_minutes=30)}, {'text': 'show Italian vegetarian recipes', 'gold': CravingParse(intent='search', cuisine='italian', diet='vegetarian')}, {'text': 'reset', 'gold': CravingParse(intent='reset')}]  # noqa: E501

def template_baseline(text):
    engine = NLU({'potato', 'onion', 'tomato'}, use_qwen=False)
    raw = engine.extract(text)
    return CravingParse.from_mapping({'intent': raw.get('intent', 'search'), 'ingredients': raw.get('ingredients', []), 'excluded_ingredients': raw.get('excluded', []), 'cuisine': raw.get('cuisine'), 'diet': raw.get('diet'), 'max_minutes': raw.get('max_minutes')})  # noqa: E501
print('Illustrative template baseline:', evaluate_nlu(EVAL_FIXTURES, template_baseline))
print('Qwen evaluation: NOT RUN. Supply real Qwen parser and same fixtures / larger gold set.')


Illustrative template baseline: {'samples': 3, 'intent_accuracy': 1.0, 'slot_precision': 1.0, 'slot_recall': 1.0, 'slot_f1': 1.0, 'schema_validity': 1.0, 'latency_p50_ms': 0.08639972656965256, 'latency_p95_ms': 1.2452001683413982, 'errors': []}
Qwen evaluation: NOT RUN. Supply real Qwen parser and same fixtures / larger gold set.


## 20. Contract smoke tests and release gate

Tests below run without model downloads or a database. They do **not** replace `pytest` integration tests against FastAPI + PostgreSQL, Alembic upgrade/downgrade tests, API authorization tests, Next.js E2E tests, or a weekly solver's hard-constraint verification.

In [23]:
def assert_raises(exception, fn):
    try:
        fn()
    except exception:
        return
    raise AssertionError(f'Expected {exception.__name__}')
assert_raises(ValueError, lambda: SafeProfileDraft('u', 'vegan', [], ['unknown-nut']).validate())
assert_raises(RuntimeError, lambda: profile_to_rule_context(profile_example))
assert_raises(NotImplementedError, lambda: onboarding_reference(profile_example))
assert_raises(PermissionError, lambda: seed_dish_shortlist(profile_example))
assert_raises(RuntimeError, lambda: propose_swap(sample_plan, slot='D1_LUNCH', replacement='x', expected_revision=0))  # noqa: E501
assert_raises(PermissionError, lambda: propose_swap(sample_plan, slot='D1_LUNCH', replacement='x', expected_revision=1, locked_slots=['D1_LUNCH']))  # noqa: E501
assert_raises(ValueError, lambda: parse_craving_json('{"intent":"search","unsafe":true}'))
assert allergen_eligibility(None, profile_example) == 'UNKNOWN'
print('PASS: fail-closed allergy, accepted KB, onboarding, seed approval, revision, lock and NLU contract guards')  # noqa: E501


PASS: fail-closed allergy, accepted KB, onboarding, seed approval, revision, lock and NLU contract guards


## 21. Optional local Gradio UI and data discovery

Run the original notebook setup and class cells first. The local `RAW_recipes.csv` is intentionally **not committed**. This cell uses `SAVORA_DATA_DIR` when set, then searches common repository locations. It creates a new bot and launches a Gradio ChatInterface; no Qwen download is triggered. For multi-user deployment, replace the shared bot with per-session state. For allergies and clinical restrictions, this prototype must not be used as the safety authority.

In [24]:
def locate_recipe_csv():
    starts = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    env = os.environ.get('SAVORA_DATA_DIR')
    if env:
        starts.insert(0, Path(env))
    candidates = []
    for root in starts:
        candidates.extend([root / 'RAW_recipes.csv', root / 'Food.com' / 'RAW_recipes.csv', root / 'Data' / 'Food.com' / 'RAW_recipes.csv'])  # noqa: E501
    for path in candidates:
        if path.is_file():
            return path.resolve()
    return None
LOCAL_RECIPE_CSV = locate_recipe_csv()
print('Dataset:', LOCAL_RECIPE_CSV or 'not found; set SAVORA_DATA_DIR or use explicit Path')

# Optional local demo (uncomment only after installing Gradio):
# import gradio as gr
# if LOCAL_RECIPE_CSV is not None:
#     local_bot = SavoraBot(data_dir=LOCAL_RECIPE_CSV.parent, use_qwen=False)
#     gr.ChatInterface(fn=lambda msg, history: local_bot.respond(msg)).launch()


Dataset: C:\Users\Aryaman\OneDrive\Desktop\MR2 Workshop\Savora-AI-Meal-Planner-GC-1-1\Food.com\RAW_recipes.csv


## 22. What must be merged into the actual monorepo

**Do not represent the notebook as implementation of all 11 tasks.** Work still needed outside the notebook: joint approval of canonical schema and seed step; real SQLAlchemy models/Alembic migrations; auth/onboarding/nutrition target service; approved KB loader and exact `RuleContext` mapping; torch scorer delivery; weekly CSP; authenticated craving and swap endpoints; Next.js UI; OpenAPI generation; API/E2E tests; and a real Qwen benchmark.

Suggested sequence: (1) approve contracts and taxonomy → (2) backend migrations/auth/profile → (3) KB/rules adapters → (4) craving/seed routes → (5) weekly planning and swaps → (6) UI + generated contracts → (7) integration and E2E evaluation. Keep the existing PR as a notebook/reference contribution and create separate reviewable PRs for production modules.